In [1]:
# ============================================================
# Validation Step 1 (Automated Validation) — schema-safe
# Adjusted to add controller-state visibility and controlled subset flag
#
# Reads from:
#   D:\0-Data_Mar16_V16.5\MainDataset.csv
#   D:\0-Data_Mar16_V16.5\run_steps_v16_stage3_breakdown.csv
#   (or auto-extracts from run_steps_v16_stage3_breakdown.zip)
#
# Writes outputs to:
#   D:\0-Data_Mar16_V16.5\Validation_Step_1
#
# New additions
# - attempt field included in outputs (if present)
# - controller fields included in outputs (if present)
# - controlled_subset = Yes / No
# - summary by controlled_subset
# - style x controlled_subset summary
# ============================================================

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
BASE_DIR = Path(r"D:\0-Data_Mar16_V16.5")
OUT_DIR = BASE_DIR / "Validation_Step_1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_PATH = BASE_DIR / "MainDataset.csv"
STEPS_CSV_PATH = BASE_DIR / "run_steps_v16_stage3_breakdown.csv"
STEPS_ZIP_PATH = BASE_DIR / "run_steps_v16_stage3_breakdown.zip"

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def to_dt(s):
    return pd.to_datetime(s, errors="coerce", utc=True)

def safe_series(df, col, dtype=None, fill_value=np.nan):
    if col in df.columns:
        return df[col]
    return pd.Series(fill_value, index=df.index, dtype=dtype)

def first_present_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def get_first_present_series(df, candidates, dtype=None, fill_value=np.nan):
    col = first_present_col(df, candidates)
    if col is not None:
        return df[col], col
    return pd.Series(fill_value, index=df.index, dtype=dtype), None

def canon_style_token(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    mapping = {
        "community": "Community",
        "custom": "Custom",
        "third-party": "Third-Party",
        "third party": "Third-Party",
        "gmd": "GMD",
        "real-device": "Real-Device",
        "real device": "Real-Device",
    }
    return mapping.get(s.lower(), s)

def normalize_name(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    for ch in ['"', "'", "[", "]", "(", ")", "{", "}", ":", ";", "|"]:
        s = s.replace(ch, " ")
    s = " ".join(s.split())
    return s

def bucket_ok_missing(series_bool, valid_mask):
    out = pd.Series("missing", index=series_bool.index, dtype="object")
    out.loc[valid_mask] = "mismatch"
    out.loc[valid_mask & series_bool.fillna(False)] = "ok"
    return out

def summarize_flags(df_flags, flag_cols):
    rows = []
    n = len(df_flags)
    for c in flag_cols:
        vc = df_flags[c].fillna("missing").value_counts(dropna=False)
        rows.append({
            "check_name": c,
            "rows_total": n,
            "ok_count": int(vc.get("ok", 0)),
            "mismatch_count": int(vc.get("mismatch", 0)),
            "missing_count": int(vc.get("missing", 0)),
            "ok_pct": round(100 * vc.get("ok", 0) / n, 4) if n else 0.0,
            "mismatch_pct": round(100 * vc.get("mismatch", 0) / n, 4) if n else 0.0,
            "missing_pct": round(100 * vc.get("missing", 0) / n, 4) if n else 0.0,
        })
    return pd.DataFrame(rows)

def to_bool_loose(s):
    """
    Robust boolean coercion for mixed schema fields.
    True values: 1, true, yes, y, complete, completed, ok
    False values: 0, false, no, n, incomplete, missing, failed
    """
    x = s.astype(str).str.strip().str.lower()
    true_set = {"1", "true", "yes", "y", "complete", "completed", "ok"}
    false_set = {"0", "false", "no", "n", "incomplete", "missing", "failed"}
    out = pd.Series(np.nan, index=s.index, dtype="object")
    out.loc[x.isin(true_set)] = True
    out.loc[x.isin(false_set)] = False
    # preserve actual booleans if they got stringified strangely
    out.loc[s.eq(True)] = True
    out.loc[s.eq(False)] = False
    return out

# ------------------------------------------------------------
# Read inputs
# ------------------------------------------------------------
print("Reading MainDataset...")
main_df = pd.read_csv(MAIN_PATH, low_memory=False)
print("MainDataset shape:", main_df.shape)

if STEPS_CSV_PATH.exists():
    steps_path_to_use = STEPS_CSV_PATH
elif STEPS_ZIP_PATH.exists():
    print("Extracting step breakdown CSV from zip...")
    with zipfile.ZipFile(STEPS_ZIP_PATH, "r") as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise FileNotFoundError("No CSV found inside run_steps_v16_stage3_breakdown.zip")
        first_csv = csv_names[0]
        zf.extract(first_csv, OUT_DIR)
        extracted = OUT_DIR / first_csv
        if extracted.name != STEPS_CSV_PATH.name:
            target = OUT_DIR / STEPS_CSV_PATH.name
            extracted.replace(target)
            steps_path_to_use = target
        else:
            steps_path_to_use = extracted
else:
    raise FileNotFoundError(
        f"Could not find either:\n- {STEPS_CSV_PATH}\n- {STEPS_ZIP_PATH}"
    )

print("Reading step breakdown...")
steps_df = pd.read_csv(steps_path_to_use, low_memory=False)
print("Step breakdown shape:", steps_df.shape)

# ------------------------------------------------------------
# Basic normalization
# ------------------------------------------------------------
if "style" in main_df.columns:
    main_df["style"] = main_df["style"].map(canon_style_token)

if "target_style" in steps_df.columns:
    steps_df["target_style"] = steps_df["target_style"].map(canon_style_token)

main_df["run_id"] = safe_series(main_df, "run_id", dtype="object", fill_value="").astype(str)
steps_df["run_id"] = safe_series(steps_df, "run_id", dtype="object", fill_value="").astype(str)

steps_df["step_name_norm"] = safe_series(steps_df, "step_name", dtype="object", fill_value="").map(normalize_name)
steps_df["job_name_norm"] = safe_series(steps_df, "job_name", dtype="object", fill_value="").map(normalize_name)

steps_df["selected_invocation_cutpoint"] = (
    safe_series(steps_df, "selected_invocation_cutpoint", dtype="object", fill_value="false")
    .astype(str).str.lower().eq("true")
)
steps_df["selected_execution_end_cutpoint"] = (
    safe_series(steps_df, "selected_execution_end_cutpoint", dtype="object", fill_value="false")
    .astype(str).str.lower().eq("true")
)

# ------------------------------------------------------------
# Step-level selected cutpoint indexes
# ------------------------------------------------------------
inv_steps = steps_df.loc[steps_df["selected_invocation_cutpoint"]].copy()
end_steps = steps_df.loc[steps_df["selected_execution_end_cutpoint"]].copy()

inv_grp = (
    inv_steps.groupby(["full_name", "run_id", "target_style"], dropna=False)
    .size().rename("selected_invocation_cutpoint_count").reset_index()
    if len(inv_steps) else
    pd.DataFrame(columns=["full_name", "run_id", "target_style", "selected_invocation_cutpoint_count"])
)

end_grp = (
    end_steps.groupby(["full_name", "run_id", "target_style"], dropna=False)
    .size().rename("selected_execution_end_cutpoint_count").reset_index()
    if len(end_steps) else
    pd.DataFrame(columns=["full_name", "run_id", "target_style", "selected_execution_end_cutpoint_count"])
)

inv_one = (
    inv_steps.sort_values(["full_name", "run_id", "target_style", "started_at"])
    .drop_duplicates(["full_name", "run_id", "target_style"], keep="first")
    [[c for c in [
        "full_name", "run_id", "target_style",
        "step_name", "job_name", "started_at", "completed_at",
        "job_ordinal_in_run", "step_ordinal_in_job"
    ] if c in inv_steps.columns]]
    .rename(columns={
        "step_name": "src_inv_step_name",
        "job_name": "src_inv_job_name",
        "started_at": "src_inv_started_at",
        "completed_at": "src_inv_completed_at",
        "job_ordinal_in_run": "src_inv_job_ordinal_in_run",
        "step_ordinal_in_job": "src_inv_step_ordinal_in_job",
    })
    if len(inv_steps) else
    pd.DataFrame(columns=[
        "full_name", "run_id", "target_style",
        "src_inv_step_name", "src_inv_job_name", "src_inv_started_at",
        "src_inv_completed_at", "src_inv_job_ordinal_in_run", "src_inv_step_ordinal_in_job"
    ])
)

end_one = (
    end_steps.sort_values(["full_name", "run_id", "target_style", "completed_at"])
    .drop_duplicates(["full_name", "run_id", "target_style"], keep="last")
    [[c for c in [
        "full_name", "run_id", "target_style",
        "step_name", "job_name", "started_at", "completed_at",
        "job_ordinal_in_run", "step_ordinal_in_job"
    ] if c in end_steps.columns]]
    .rename(columns={
        "step_name": "src_end_step_name",
        "job_name": "src_end_job_name",
        "started_at": "src_end_started_at",
        "completed_at": "src_end_completed_at",
        "job_ordinal_in_run": "src_end_job_ordinal_in_run",
        "step_ordinal_in_job": "src_end_step_ordinal_in_job",
    })
    if len(end_steps) else
    pd.DataFrame(columns=[
        "full_name", "run_id", "target_style",
        "src_end_step_name", "src_end_job_name", "src_end_started_at",
        "src_end_completed_at", "src_end_job_ordinal_in_run", "src_end_step_ordinal_in_job"
    ])
)

# ------------------------------------------------------------
# Join step-source evidence into MainDataset
# ------------------------------------------------------------
val_df = main_df.copy()

if len(inv_grp):
    val_df = val_df.merge(
        inv_grp,
        left_on=["full_name", "run_id", "style"],
        right_on=["full_name", "run_id", "target_style"],
        how="left",
    )
    val_df = val_df.drop(columns=[c for c in ["target_style"] if c in val_df.columns])

if len(end_grp):
    val_df = val_df.merge(
        end_grp,
        left_on=["full_name", "run_id", "style"],
        right_on=["full_name", "run_id", "target_style"],
        how="left",
    )
    val_df = val_df.drop(columns=[c for c in ["target_style"] if c in val_df.columns])

if len(inv_one):
    val_df = val_df.merge(
        inv_one,
        left_on=["full_name", "run_id", "style"],
        right_on=["full_name", "run_id", "target_style"],
        how="left",
    )
    val_df = val_df.drop(columns=[c for c in ["target_style"] if c in val_df.columns])

if len(end_one):
    val_df = val_df.merge(
        end_one,
        left_on=["full_name", "run_id", "style"],
        right_on=["full_name", "run_id", "target_style"],
        how="left",
    )
    val_df = val_df.drop(columns=[c for c in ["target_style"] if c in val_df.columns])

# ------------------------------------------------------------
# Controlled subset fields
# ------------------------------------------------------------
attempt_series, attempt_col = get_first_present_series(
    val_df,
    ["attempt", "run_attempt", "study_attempt"],
    dtype="object",
    fill_value=np.nan
)
val_df["attempt_resolved"] = to_num(attempt_series)

run_ctrl_series, run_ctrl_col = get_first_present_series(
    val_df,
    [
        "controller_run_verdict_complete",
        "run_verdict_complete",
        "study_controller_run_verdict_complete",
        "controller_run_complete",
    ],
    dtype="object",
    fill_value=np.nan
)
inst_ctrl_series, inst_ctrl_col = get_first_present_series(
    val_df,
    [
        "controller_instrumentation_verdict_complete",
        "instrumentation_verdict_complete",
        "study_controller_instrumentation_verdict_complete",
        "controller_instrumentation_complete",
    ],
    dtype="object",
    fill_value=np.nan
)

val_df["controller_run_verdict_complete_resolved"] = to_bool_loose(run_ctrl_series)
val_df["controller_instrumentation_verdict_complete_resolved"] = to_bool_loose(inst_ctrl_series)

controlled_subset_yes = (
    val_df["attempt_resolved"].eq(1) &
    val_df["controller_run_verdict_complete_resolved"].eq(True) &
    val_df["controller_instrumentation_verdict_complete_resolved"].eq(True)
)

val_df["controlled_subset"] = np.where(controlled_subset_yes, "Yes", "No")

# ------------------------------------------------------------
# Datetime conversion
# ------------------------------------------------------------
run_start_dt = to_dt(safe_series(val_df, "study_run_boundary_start_at"))
run_end_dt = to_dt(safe_series(val_df, "study_run_boundary_end_at"))
inv_start_dt = to_dt(safe_series(val_df, "study_matched_invocation_step_started_at"))
exec_end_dt = to_dt(safe_series(val_df, "study_invocation_execution_end_step_completed_at"))
win_start_dt = to_dt(safe_series(val_df, "study_invocation_execution_window_started_at"))
win_end_dt = to_dt(safe_series(val_df, "study_invocation_execution_window_ended_at"))

# ------------------------------------------------------------
# Structural checks
# ------------------------------------------------------------
dup_mask = val_df.duplicated(subset=["full_name", "run_id", "style"], keep=False)
val_df["v1_key_uniqueness_flag"] = np.where(dup_mask, "mismatch", "ok")

required_present = (
    safe_series(val_df, "full_name").notna() &
    safe_series(val_df, "run_id").notna() &
    safe_series(val_df, "style").notna() &
    safe_series(val_df, "workflow_id").notna() &
    safe_series(val_df, "workflow_identifier").notna()
)
val_df["v2_required_ids_flag"] = np.where(required_present, "ok", "mismatch")

# ------------------------------------------------------------
# Temporal ordering checks
# ------------------------------------------------------------
have_run_bounds = run_start_dt.notna() & run_end_dt.notna()
val_df["v3_run_bounds_order_flag"] = bucket_ok_missing(run_start_dt.le(run_end_dt), have_run_bounds)

have_window_bounds = win_start_dt.notna() & win_end_dt.notna()
val_df["v4_window_bounds_order_flag"] = bucket_ok_missing(win_start_dt.le(win_end_dt), have_window_bounds)

have_cutpoints = run_start_dt.notna() & inv_start_dt.notna() & exec_end_dt.notna() & run_end_dt.notna()
temporal_ok = (run_start_dt <= inv_start_dt) & (inv_start_dt <= exec_end_dt) & (exec_end_dt <= run_end_dt)
val_df["v5_cutpoint_temporal_order_flag"] = bucket_ok_missing(temporal_ok, have_cutpoints)

have_window_with_run = have_window_bounds & have_run_bounds
window_in_run_ok = (run_start_dt <= win_start_dt) & (win_end_dt <= run_end_dt)
val_df["v6_window_inside_run_flag"] = bucket_ok_missing(window_in_run_ok, have_window_with_run)

# ------------------------------------------------------------
# Duration checks
# ------------------------------------------------------------
run_dur = to_num(safe_series(val_df, "study_run_duration_seconds"))
pre_dur = to_num(safe_series(val_df, "study_pre_invocation_selected_stage3_seconds"))
win_dur = to_num(safe_series(val_df, "study_invocation_execution_window_selected_stage3_seconds"))
post_dur = to_num(safe_series(val_df, "study_post_invocation_selected_stage3_seconds"))

have_run_dur = run_dur.notna()
val_df["v7_run_duration_nonnegative_flag"] = bucket_ok_missing(run_dur.ge(0), have_run_dur)

have_l2 = pre_dur.notna() & win_dur.notna() & post_dur.notna()
val_df["v8_layer2_nonnegative_flag"] = bucket_ok_missing(pre_dur.ge(0) & win_dur.ge(0) & post_dur.ge(0), have_l2)

have_component_vs_run = have_run_dur & pre_dur.notna() & win_dur.notna() & post_dur.notna()
comp_vs_run_ok = pre_dur.le(run_dur) & win_dur.le(run_dur) & post_dur.le(run_dur)
val_df["v9_components_le_run_flag"] = bucket_ok_missing(comp_vs_run_ok, have_component_vs_run)

l1_a = to_num(safe_series(val_df, "study_layer1_time_to_instrumentation_envelope_seconds"))
l1_b = to_num(safe_series(val_df, "study_layer1_instrumentation_job_envelope_seconds"))
l1_c = to_num(safe_series(val_df, "study_layer1_post_instrumentation_tail_seconds"))

have_l1 = have_run_dur & l1_a.notna() & l1_b.notna() & l1_c.notna()
l1_sum = l1_a.fillna(0) + l1_b.fillna(0) + l1_c.fillna(0)
l1_diff = l1_sum - run_dur
l1_ok = l1_diff.abs().le(1e-9)
val_df["v10_layer1_sum_to_run_flag"] = bucket_ok_missing(l1_ok, have_l1)
val_df["v10_layer1_sum_diff_seconds"] = l1_diff.where(have_l1, np.nan)

decomp_diff = to_num(safe_series(val_df, "study_window_decomp_diff_seconds"))
have_decomp = decomp_diff.notna()
decomp_ok = decomp_diff.abs().le(1e-9)
val_df["v11_window_decomposition_flag"] = bucket_ok_missing(decomp_ok, have_decomp)

# ------------------------------------------------------------
# Cutpoint recomputation checks
# ------------------------------------------------------------
pre_from_cut = (inv_start_dt - run_start_dt).dt.total_seconds()
win_from_cut = (exec_end_dt - inv_start_dt).dt.total_seconds()
post_from_cut = (run_end_dt - exec_end_dt).dt.total_seconds()

have_cut_recomp = have_cutpoints & pre_dur.notna() & win_dur.notna() & post_dur.notna()

pre_diff = pre_from_cut - pre_dur
win_diff = win_from_cut - win_dur
post_diff = post_from_cut - post_dur

val_df["v12_pre_cutpoint_diff_seconds"] = pre_diff.where(have_cut_recomp, np.nan)
val_df["v13_window_cutpoint_diff_seconds"] = win_diff.where(have_cut_recomp, np.nan)
val_df["v14_post_cutpoint_diff_seconds"] = post_diff.where(have_cut_recomp, np.nan)

val_df["v12_pre_cutpoint_flag"] = bucket_ok_missing(pre_diff.abs().le(1e-9), have_cut_recomp)
val_df["v13_window_cutpoint_flag"] = bucket_ok_missing(win_diff.abs().le(1e-9), have_cut_recomp)
val_df["v14_post_cutpoint_flag"] = bucket_ok_missing(post_diff.abs().le(1e-9), have_cut_recomp)

# ------------------------------------------------------------
# Step-source cross-checks
# ------------------------------------------------------------
inv_count = to_num(safe_series(val_df, "selected_invocation_cutpoint_count"))
end_count = to_num(safe_series(val_df, "selected_execution_end_cutpoint_count"))

have_inv_source = inv_count.notna()
have_end_source = end_count.notna()

val_df["v15_selected_invocation_cutpoint_count_flag"] = bucket_ok_missing(inv_count.eq(1), have_inv_source)
val_df["v16_selected_execution_end_cutpoint_count_flag"] = bucket_ok_missing(end_count.eq(1), have_end_source)

inv_name_match = safe_series(val_df, "study_matched_invocation_step_name", dtype="object", fill_value="").map(normalize_name) == safe_series(val_df, "src_inv_step_name", dtype="object", fill_value="").map(normalize_name)
inv_job_match = safe_series(val_df, "study_matched_invocation_job_name", dtype="object", fill_value="").map(normalize_name) == safe_series(val_df, "src_inv_job_name", dtype="object", fill_value="").map(normalize_name)
inv_time_match = safe_series(val_df, "study_matched_invocation_step_started_at", dtype="object", fill_value="").astype(str) == safe_series(val_df, "src_inv_started_at", dtype="object", fill_value="").astype(str)
have_inv_match = safe_series(val_df, "src_inv_step_name").notna()

val_df["v17_invocation_step_match_flag"] = bucket_ok_missing(inv_name_match & inv_job_match & inv_time_match, have_inv_match)

end_name_match = safe_series(val_df, "study_invocation_execution_end_step_name", dtype="object", fill_value="").map(normalize_name) == safe_series(val_df, "src_end_step_name", dtype="object", fill_value="").map(normalize_name)
end_job_match = safe_series(val_df, "study_invocation_execution_end_job_name", dtype="object", fill_value="").map(normalize_name) == safe_series(val_df, "src_end_job_name", dtype="object", fill_value="").map(normalize_name)
end_time_match = safe_series(val_df, "study_invocation_execution_end_step_completed_at", dtype="object", fill_value="").astype(str) == safe_series(val_df, "src_end_completed_at", dtype="object", fill_value="").astype(str)
have_end_match = safe_series(val_df, "src_end_step_name").notna()

val_df["v18_execution_end_step_match_flag"] = bucket_ok_missing(end_name_match & end_job_match & end_time_match, have_end_match)

# ------------------------------------------------------------
# Controller / scope / coverage checks
# ------------------------------------------------------------
val_df["v19_style_scope_flag"] = np.where(
    safe_series(val_df, "style").isin(["Community", "Custom", "Third-Party", "GMD"]),
    "ok",
    "mismatch"
)
val_df["v20_base_flag_presence"] = np.where(safe_series(val_df, "Base").notna(), "ok", "mismatch")
val_df["v21_robust_flag_presence"] = np.where(safe_series(val_df, "Robust").notna(), "ok", "mismatch")

# ------------------------------------------------------------
# Flag columns
# ------------------------------------------------------------
flag_cols = [
    "v1_key_uniqueness_flag",
    "v2_required_ids_flag",
    "v3_run_bounds_order_flag",
    "v4_window_bounds_order_flag",
    "v5_cutpoint_temporal_order_flag",
    "v6_window_inside_run_flag",
    "v7_run_duration_nonnegative_flag",
    "v8_layer2_nonnegative_flag",
    "v9_components_le_run_flag",
    "v10_layer1_sum_to_run_flag",
    "v11_window_decomposition_flag",
    "v12_pre_cutpoint_flag",
    "v13_window_cutpoint_flag",
    "v14_post_cutpoint_flag",
    "v15_selected_invocation_cutpoint_count_flag",
    "v16_selected_execution_end_cutpoint_count_flag",
    "v17_invocation_step_match_flag",
    "v18_execution_end_step_match_flag",
    "v19_style_scope_flag",
    "v20_base_flag_presence",
    "v21_robust_flag_presence",
]

# ------------------------------------------------------------
# Output tables
# ------------------------------------------------------------
controller_cols_for_output = [
    c for c in [
        attempt_col,
        run_ctrl_col,
        inst_ctrl_col,
    ] if c is not None
]

resolved_controller_cols = [
    "attempt_resolved",
    "controller_run_verdict_complete_resolved",
    "controller_instrumentation_verdict_complete_resolved",
    "controlled_subset",
]

record_cols = (
    [c for c in ["full_name", "run_id", "style", "workflow_identifier", "workflow_id"] if c in val_df.columns] +
    controller_cols_for_output +
    resolved_controller_cols +
    flag_cols +
    [c for c in [
        "v10_layer1_sum_diff_seconds",
        "v12_pre_cutpoint_diff_seconds",
        "v13_window_cutpoint_diff_seconds",
        "v14_post_cutpoint_diff_seconds",
        "study_timing_consistency_flag",
        "study_cutpoint_consistency_flag",
        "study_temporal_order_flag",
    ] if c in val_df.columns]
)

record_flags = val_df[record_cols].copy()
record_flags.to_csv(OUT_DIR / "validation_step1_record_flags.csv", index=False)

summary_df = summarize_flags(val_df, flag_cols)
summary_df.to_csv(OUT_DIR / "validation_step1_summary.csv", index=False)

style_summary_parts = []
if "style" in val_df.columns:
    for style_name, g in val_df.groupby("style", dropna=False):
        s = summarize_flags(g, flag_cols)
        s.insert(0, "style", style_name)
        style_summary_parts.append(s)
style_summary_df = pd.concat(style_summary_parts, ignore_index=True) if style_summary_parts else pd.DataFrame()
style_summary_df.to_csv(OUT_DIR / "validation_step1_style_summary.csv", index=False)

controlled_parts = []
for ctrl_name, g in val_df.groupby("controlled_subset", dropna=False):
    s = summarize_flags(g, flag_cols)
    s.insert(0, "controlled_subset", ctrl_name)
    controlled_parts.append(s)
controlled_summary_df = pd.concat(controlled_parts, ignore_index=True) if controlled_parts else pd.DataFrame()
controlled_summary_df.to_csv(OUT_DIR / "validation_step1_controlled_subset_summary.csv", index=False)

style_controlled_parts = []
if "style" in val_df.columns:
    for (style_name, ctrl_name), g in val_df.groupby(["style", "controlled_subset"], dropna=False):
        s = summarize_flags(g, flag_cols)
        s.insert(0, "controlled_subset", ctrl_name)
        s.insert(0, "style", style_name)
        style_controlled_parts.append(s)
style_controlled_summary_df = (
    pd.concat(style_controlled_parts, ignore_index=True)
    if style_controlled_parts else pd.DataFrame()
)
style_controlled_summary_df.to_csv(
    OUT_DIR / "validation_step1_style_controlled_subset_summary.csv",
    index=False
)

mismatch_mask = (
    val_df["v12_pre_cutpoint_flag"].eq("mismatch") |
    val_df["v13_window_cutpoint_flag"].eq("mismatch") |
    val_df["v14_post_cutpoint_flag"].eq("mismatch") |
    val_df["v5_cutpoint_temporal_order_flag"].eq("mismatch") |
    val_df["v6_window_inside_run_flag"].eq("mismatch")
)

cutpoint_issue_cols = [c for c in [
    "full_name", "run_id", "style",
    "workflow_identifier", "workflow_id",
    attempt_col,
    run_ctrl_col,
    inst_ctrl_col,
    "attempt_resolved",
    "controller_run_verdict_complete_resolved",
    "controller_instrumentation_verdict_complete_resolved",
    "controlled_subset",
    "study_run_boundary_start_at", "study_run_boundary_end_at",
    "study_matched_invocation_step_name", "study_matched_invocation_job_name",
    "study_matched_invocation_step_started_at",
    "study_invocation_execution_end_step_name", "study_invocation_execution_end_job_name",
    "study_invocation_execution_end_step_completed_at",
    "study_invocation_execution_window_started_at",
    "study_invocation_execution_window_ended_at",
    "study_pre_invocation_selected_stage3_seconds",
    "study_invocation_execution_window_selected_stage3_seconds",
    "study_post_invocation_selected_stage3_seconds",
    "v12_pre_cutpoint_diff_seconds",
    "v13_window_cutpoint_diff_seconds",
    "v14_post_cutpoint_diff_seconds",
    "v5_cutpoint_temporal_order_flag",
    "v6_window_inside_run_flag",
] if c in val_df.columns]

val_df.loc[mismatch_mask, cutpoint_issue_cols].to_csv(
    OUT_DIR / "validation_step1_cutpoint_mismatches.csv", index=False
)

step_match_issue_mask = (
    val_df["v15_selected_invocation_cutpoint_count_flag"].eq("mismatch") |
    val_df["v16_selected_execution_end_cutpoint_count_flag"].eq("mismatch") |
    val_df["v17_invocation_step_match_flag"].eq("mismatch") |
    val_df["v18_execution_end_step_match_flag"].eq("mismatch")
)

step_issue_cols = [c for c in [
    "full_name", "run_id", "style",
    "workflow_identifier", "workflow_id",
    attempt_col,
    run_ctrl_col,
    inst_ctrl_col,
    "attempt_resolved",
    "controller_run_verdict_complete_resolved",
    "controller_instrumentation_verdict_complete_resolved",
    "controlled_subset",
    "selected_invocation_cutpoint_count",
    "selected_execution_end_cutpoint_count",
    "study_matched_invocation_step_name", "study_matched_invocation_job_name", "study_matched_invocation_step_started_at",
    "src_inv_step_name", "src_inv_job_name", "src_inv_started_at",
    "study_invocation_execution_end_step_name", "study_invocation_execution_end_job_name", "study_invocation_execution_end_step_completed_at",
    "src_end_step_name", "src_end_job_name", "src_end_completed_at",
    "v15_selected_invocation_cutpoint_count_flag",
    "v16_selected_execution_end_cutpoint_count_flag",
    "v17_invocation_step_match_flag",
    "v18_execution_end_step_match_flag",
] if c in val_df.columns]

val_df.loc[step_match_issue_mask, step_issue_cols].to_csv(
    OUT_DIR / "validation_step1_step_match_issues.csv", index=False
)

# Optional: direct controlled-subset-only mismatch file for fastest audit
val_df.loc[
    mismatch_mask & val_df["controlled_subset"].eq("Yes"),
    cutpoint_issue_cols
].to_csv(
    OUT_DIR / "validation_step1_cutpoint_mismatches_controlled_subset_only.csv",
    index=False
)

# ------------------------------------------------------------
# Console output
# ------------------------------------------------------------
print("\nResolved controller columns:")
print(" - attempt:", attempt_col)
print(" - run verdict complete:", run_ctrl_col)
print(" - instrumentation verdict complete:", inst_ctrl_col)

print("\nSaved outputs to:", OUT_DIR)
print(" - validation_step1_record_flags.csv")
print(" - validation_step1_summary.csv")
print(" - validation_step1_style_summary.csv")
print(" - validation_step1_controlled_subset_summary.csv")
print(" - validation_step1_style_controlled_subset_summary.csv")
print(" - validation_step1_cutpoint_mismatches.csv")
print(" - validation_step1_cutpoint_mismatches_controlled_subset_only.csv")
print(" - validation_step1_step_match_issues.csv")

print("\nTop-level summary:")
print(summary_df)

print("\nControlled subset breakdown:")
print(
    val_df["controlled_subset"]
    .value_counts(dropna=False)
    .rename_axis("controlled_subset")
    .reset_index(name="rows")
)

Reading MainDataset...
MainDataset shape: (8900, 124)
Reading step breakdown...
Step breakdown shape: (2857775, 41)

Resolved controller columns:
 - attempt: run_attempt
 - run verdict complete: controller_run_verdict_complete
 - instrumentation verdict complete: None

Saved outputs to: D:\0-Data_Mar16_V16.5\Validation_Step_1
 - validation_step1_record_flags.csv
 - validation_step1_summary.csv
 - validation_step1_style_summary.csv
 - validation_step1_controlled_subset_summary.csv
 - validation_step1_style_controlled_subset_summary.csv
 - validation_step1_cutpoint_mismatches.csv
 - validation_step1_cutpoint_mismatches_controlled_subset_only.csv
 - validation_step1_step_match_issues.csv

Top-level summary:
                                        check_name  rows_total  ok_count  \
0                           v1_key_uniqueness_flag        8900      8900   
1                             v2_required_ids_flag        8900      8900   
2                         v3_run_bounds_order_flag        